In [1]:
import os
os.chdir(r"..\models\..")

import math
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import pickle

from tokenizer import Tokenizer
from transformer.transformer import Transformer

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, random_split

In [2]:
# Training Tokenizer
data = "data/translation_en_es_data.csv"
df = pd.read_csv(data)

df = df.sample(n=30_000, random_state=42)

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Dropping missing values >:(
train_df = train_df.dropna(subset=["English", "Spanish"])
val_df = val_df.dropna(subset=["English", "Spanish"])

tokenizer = Tokenizer(vocab_size=100_000)

texts = list(train_df["English"]) + list(train_df["Spanish"])
tokenizer.fit(texts)

vocab_size = len(tokenizer.word_to_idx)
print(f"Vocabulary Size: {vocab_size}")

with open("models/translator_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

Vocabulary Size: 40889


In [3]:
class TranslationDataset(Dataset):
    def __init__(self, df, tokenizer, src_max_len=64, trg_max_len=64):
        self.src_texts = df["English"].tolist()
        self.trg_texts = df["Spanish"].tolist()
        self.tokenizer = tokenizer
        self.src_max_len = src_max_len
        self.trg_max_len = trg_max_len

    def __len__(self):
        return len(self.src_texts)

    def __getitem__(self, idx):
        src = self.tokenizer.transform(self.src_texts[idx])
        trg = self.tokenizer.transform(self.trg_texts[idx])

        trg = ([self.tokenizer.word_to_idx["<SOS>"]] + trg + [self.tokenizer.word_to_idx["<EOS>"]])

        src = self.tokenizer.pad_sequence([src], self.src_max_len)[0]
        trg = self.tokenizer.pad_sequence([trg], self.trg_max_len)[0]

        return torch.tensor(src, dtype=torch.long), torch.tensor(trg, dtype=torch.long)

In [4]:
# Plotting Metrics Function
import matplotlib.pyplot as plt

def plot_metrics(train_losses, val_losses, perplexities):
    epochs = range(1, len(val_losses) + 1)

    plt.figure(figsize=(12, 5))

    # Loss Plotting
    plt.subplot(2, 2, 1)
    plt.plot(epochs, train_losses, label="Train Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training Loss")
    plt.grid()
    plt.legend()

    # Accuracy Plotting
    plt.subplot(2, 2, 2)
    plt.plot(epochs, val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Validation Loss")
    plt.grid()
    plt.legend()
    
    # Perplexity Plotting
    plt.subplot(2, 2, 3)
    plt.plot(epochs, perplexities, label="Perplexity")
    plt.xlabel("Epoch")
    plt.ylabel("Perplexity")
    plt.title("Perplexity per Epoch")
    plt.grid()
    plt.legend()

    # BLEU Plotting
    plt.subplot(2, 2, 4)
    plt.plot(epochs, perplexities, label="BLEU Score")
    plt.xlabel("Epoch")
    plt.ylabel("BLEU score")
    plt.title("BLEU score per Epoch")
    plt.grid()
    plt.legend()
    
    plt.tight_layout()
    plt.show()

In [5]:
from nltk.translate.bleu_score import SmoothingFunction
from nltk.translate.bleu_score import corpus_bleu

def greedy_decode(model, src, tokenizer, max_len, device):
    model.eval()
    src = src.to(device)
    trg_tokens = [tokenizer.word_to_idx["<SOS>"]]

    for _ in range(max_len):
        trg_tensor = torch.tensor(trg_tokens, dtype=torch.long).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(src, trg_tensor)
        
        next_token = output[:, -1, :].argmax(dim=-1).item()

        trg_tokens.append(next_token)

        if next_token == tokenizer.word_to_idx["<EOS>"]:
            break

    return trg_tokens

In [6]:
# Training the model
train_data = TranslationDataset(train_df, tokenizer)
val_data = TranslationDataset(val_df, tokenizer)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
src_pad_idx = tokenizer.word_to_idx["<PAD>"]
trg_pad_idx = tokenizer.word_to_idx["<PAD>"]

model = Transformer(
    vocab_size=vocab_size,
    src_pad_idx=src_pad_idx,
    trg_pad_idx=trg_pad_idx,
    embedding_dim=128,
    num_layers=2,
    num_heads=4,
    d_ff=256,
    max_len=64,
    dropout=0.1,
    device=device
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=trg_pad_idx)
optimizer = optim.Adam(model.parameters(), lr=3e-4)

def train(model, train_loader, val_loader, criterion, optimizer, device, epochs=100):

    best_val_loss = float("inf")
    smooth = SmoothingFunction().method1

    train_losses = []
    val_losses = []
    perplexities = []
    blue_scores = []

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for src, trg in train_loader:

            src = src.to(device)
            trg = trg.to(device)
            
            # Loss 
            trg_input = trg[:, :-1]
            trg_target = trg[:, 1:]

            optimizer.zero_grad()
            
            output = model(src, trg_input)

            loss = criterion(output.reshape(-1, output.shape[-1]), trg_target.reshape(-1))
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            

        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0
        references = []
        hypotheses = []
        
        with torch.no_grad():
            for src, trg in val_loader:
                src = src.to(device)
                trg = trg.to(device)

                trg_input = trg[:, :-1]
                trg_target = trg[:, 1:]

                output = model(src, trg_input)
                loss = criterion(output.reshape(-1, output.shape[-1]), trg_target.reshape(-1))
                val_loss += loss.item()

                for i in range(src.size(0)):

                    generated = greedy_decode(
                        model,
                        src[i].unsqueeze(0),
                        tokenizer,
                        max_len=64,
                        device=device
                    )

                    # Removing SOS/EOS/PAD from generated
                    generated = [
                        token for token in generated
                        if token not in (
                            tokenizer.word_to_idx["<SOS>"],
                            tokenizer.word_to_idx["<EOS>"],
                            tokenizer.word_to_idx["<PAD>"]
                        )
                    ]

                    # Removing PAD from reference
                    reference = [
                        token.item() for token in trg[i]
                        if token.item() not in (
                            tokenizer.word_to_idx["<SOS>"],
                            tokenizer.word_to_idx["<EOS>"],
                            tokenizer.word_to_idx["<PAD>"]
                        )
                    ]

                    references.append([reference])
                    hypotheses.append(generated)

        val_loss /= len(val_loader)
        perplexity = math.exp(val_loss)
        blue_score = corpus_bleu(references, hypotheses, smoothing_function=smooth)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        perplexities.append(perplexity)
        blue_scores.append(blue_score)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Perplexity: {perplexity:.2f}, BLEU: {blue_score:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "models/translator_best_model.pt")

    plot_metrics(train_losses, val_losses, perplexities, blue_scores)

print("\nCUDA available:", torch.cuda.is_available())
print("CUDA device name:", torch.cuda.get_device_name(torch.cuda.current_device()))
train(model, train_loader, val_loader, criterion, optimizer, device, epochs=100)



CUDA available: True
CUDA device name: NVIDIA GeForce GTX 1660


RuntimeError: stack expects each tensor to be equal size, but got [3] at entry 0 and [57] at entry 1

In [ ]:
# Trying model
with open("models/generator_tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Transformer(
    vocab_size=vocab_size,
    src_pad_idx=src_pad_idx,
    trg_pad_idx=trg_pad_idx,
    embedding_dim=128,
    num_layers=2,
    num_heads=4,
    d_ff=256,
    max_len=64,
    dropout=0.1,
    device=device
).to(device)

model.load_state_dict(torch.load("models/translator_best_model.pt", map_location=device))

model.eval()

def translate_text(model, tokenizer, sentence, max_len=50):
    model.eval()

    src_tokens = tokenizer.transform(sentence)
    src = torch.tensor(src_tokens).unsqueeze(0).to(device)

    trg_tokens = [tokenizer.word_to_idx["<SOS>"]]
    trg = torch.tensor(trg_tokens).unsqueeze(0).to(device)

    for _ in range(max_len):
        with torch.no_grad():
            output = model(src, trg)

        next_token_logits = output[:, -1, :]
        next_token = torch.argmax(next_token_logits, dim=-1)

        trg = torch.cat([trg, next_token.unsqueeze(0)], dim=1)

        if next_token.item() == tokenizer.word_to_idx["<EOS>"]:
            break

    generated_tokens = trg.squeeze().tolist()
    words = [tokenizer.idx_to_word.get(t, "<UNK>") for t in generated_tokens]
    lines = [" ".join(words[i:i+10]) for i in range(0, len(words), 10)]

    return "\n".join(lines)

print("Translation to my sentence:\n")
print(translate_text(
    model,
    tokenizer,
    sentence="I am very happy, for you my love",
    max_len=64
))

Translation to my sentence:

<SOS> and about his problems : two to and about
to and about to and about , it that girl
to and about to and about to and about to
and about to and about to and being has the
kunyu the agricultural the location to and about the location
to
